In [2]:
# ===== 上传并解压 zip =====
import zipfile   # 处理 zip 压缩包
import glob      # 按模式查找文件

from google.colab import files
uploaded = files.upload()                    # 弹出选择框，选你的 zip
zip_name = list(uploaded.keys())[0]          # 取出上传的文件名
print("上传的文件:", zip_name)

with zipfile.ZipFile(zip_name) as z:
    print("压缩包里的内容:", z.namelist())   # 看看里面是 xlsx 还是 csv
    z.extractall('data')                     # 解压到 data 文件夹

found = glob.glob('data/**/*.xlsx', recursive=True)   # 找解压后的 xlsx
print("找到的 xlsx:", found)

Saving online+retail+ii.zip to online+retail+ii.zip
上传的文件: online+retail+ii.zip
压缩包里的内容: ['online_retail_II.xlsx']
找到的 xlsx: ['data/online_retail_II.xlsx']


In [3]:
# ===== 读取两个 sheet =====
import pandas as pd   # 处理表格数据的库

PATH = 'data/online_retail_II.xlsx'   # 解压后的真实路径

# sheet_name=None 会把所有 sheet 一次读进来，返回 {sheet名: DataFrame}
# 文件较大，读取可能需要 30 到 60 秒
sheets = pd.read_excel(PATH, sheet_name=None)

for name, df in sheets.items():
    print(f"\n--- {name} ---")
    print("形状 (行数, 列数):", df.shape)   # 行数和列数
    print(df.head(3))                        # 前 3 行，检查列名和内容


--- Year 2009-2010 ---
形状 (行数, 列数): (525461, 8)
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  

--- Year 2010-2011 ---
形状 (行数, 列数): (541910, 8)
  Invoice StockCode                         Description  Quantity  \
0  536365    85123A  WHITE HANGING HEART T-LIGHT HOLDER         6   
1  536365     71053                 WHITE METAL LANTERN         6   
2  536365    84406B      CREAM CUPID HEARTS COAT HANGER         8   

          InvoiceDate  Price  Customer ID         Country  
0 2010-12-01 08:26:00   2.55      17850.0  U

In [1]:
import glob
print(glob.glob('data/**/*.xlsx', recursive=True))

[]


In [4]:
# ===== 1.1 Combine both sheets and record the starting state =====
import pandas as pd

PATH = 'data/online_retail_II.xlsx'

# sheet_name=None reads every sheet, returning {sheet name: DataFrame}
sheets = pd.read_excel(PATH, sheet_name=None)

# Stack the two sheets into one table
# ignore_index=True renumbers the rows so the two sheets don't clash
df = pd.concat(sheets.values(), ignore_index=True)

print("Total rows after combining:", len(df))   # expect 1,067,371

# Missing values per column
# Rows with no Customer ID will be removed during cleaning
print("\nMissing values by column:")
print(df.isna().sum())

# Invoices starting with 'C' are cancellations
# astype(str) forces every invoice number to text so numeric codes don't error
cancelled = df['Invoice'].astype(str).str.startswith('C')
print("\nCancelled rows (Invoice starts with C):", cancelled.sum())

Total rows after combining: 1067371

Missing values by column:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Cancelled rows (Invoice starts with C): 19494


In [5]:
# ===== 1.2 Identify non-product stock codes =====

# Real product codes are 5 digits, sometimes with a letter suffix (e.g. 85123A).
# Anything shorter or purely alphabetic is usually a fee or an adjustment.
codes = df['StockCode'].astype(str)

# Flag codes that contain no digits at all, or are shorter than 5 characters
suspicious = codes[~codes.str.match(r'^\d{5}')]

# value_counts() lists each distinct code and how often it appears,
# sorted from most to least common
print("Distinct non-standard codes:", suspicious.nunique())
print("\nMost common ones:")
print(suspicious.value_counts().head(30))

Distinct non-standard codes: 62

Most common ones:
StockCode
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
CRUK              16
gift_0001_10      16
DCGS0076          15
TEST001           15
DCGS0003          14
gift_0001_50       8
gift_0001_40       7
B                  6
DCGS0069           6
DCGS0004           5
m                  5
gift_0001_80       4
DCGS0066N          4
DCGS0072           4
DCGS0068           3
Name: count, dtype: int64


In [6]:
# ===== 1.3 Inspect the non-standard codes with their descriptions =====

# Rebuild the flag, this time keeping the whole rows instead of just the codes
mask = ~df['StockCode'].astype(str).str.match(r'^\d{5}')
odd_rows = df[mask]

# groupby() groups the rows by stock code, then agg() calculates
# several summary figures for each group at once
summary = odd_rows.groupby(odd_rows['StockCode'].astype(str)).agg(
    rows=('Invoice', 'size'),                       # how many rows
    description=('Description', 'first'),           # the first description seen
    total_qty=('Quantity', 'sum'),                  # total units
    avg_price=('Price', 'mean')                     # average unit price
).sort_values('rows', ascending=False)

# Show all 62 rows instead of pandas' default truncated view
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
print(summary)

              rows                          description  total_qty     avg_price
StockCode                                                                       
POST          2122                              POSTAGE      10108     32.623322
DOT           1446                       DOTCOM POSTAGE       2938    223.144876
M             1421                               Manual       4607    528.776284
C2             282                             CARRIAGE        716     49.460993
D              177                             Discount      -2872     72.867288
S              104                              SAMPLES        -98     60.956731
BANK CHARGES   102                         Bank Charges        -42    359.034422
ADJUST          67  Adjustment by john on 26/01/2010 16          5    163.591343
AMAZONFEE       43                           AMAZON FEE        -35   7646.089302
DCGS0058        31                     MISO PRETTY  GUM         43      0.801935
gift_0001_20    29   Dotcomg

In [7]:
# ===== 1.4 Apply cleaning rules and log how many rows each one removes =====

# Start from the combined table
clean = df.copy()   # .copy() avoids modifying the original df
log = []            # collect a record of each rule's effect

def drop(mask, reason):
    """Drop rows where mask is True, record how many were removed."""
    global clean
    n = mask.sum()
    clean = clean[~mask]                            # keep the rows NOT flagged
    log.append({'rule': reason, 'rows_removed': n})
    return clean

# --- Rule 1: cancellations (invoice numbers starting with C) ---
drop(clean['Invoice'].astype(str).str.startswith('C'), 'Cancelled invoices')

# --- Rule 2: non-product stock codes ---
# .str.upper() makes the match case-insensitive, so both 'M' and 'm' are caught
non_products = ['POST', 'DOT', 'C2', 'C3', 'M', 'D', 'S', 'BANK CHARGES',
                'AMAZONFEE', 'CRUK', 'ADJUST', 'ADJUST2', 'B',
                'TEST001', 'TEST002', 'GIFT']
codes_upper = clean['StockCode'].astype(str).str.upper()

# .isin() checks membership in a list; .str.startswith() catches the gift vouchers
drop(codes_upper.isin(non_products) | codes_upper.str.startswith('GIFT_'),
     'Non-product codes (postage, fees, adjustments, vouchers)')

# --- Rule 3: zero or negative quantity ---
drop(clean['Quantity'] <= 0, 'Quantity <= 0')

# --- Rule 4: zero or negative price ---
drop(clean['Price'] <= 0, 'Price <= 0')

# Turn the log into a readable table
log_df = pd.DataFrame(log)
log_df['pct_of_original'] = (log_df['rows_removed'] / len(df) * 100).round(2)
print(log_df)

print("\nStarting rows:", len(df))
print("Rows after cleaning:", len(clean))
print("Total removed:", len(df) - len(clean),
      f"({(len(df) - len(clean)) / len(df) * 100:.1f}%)")

# --- Second dataset: customer-level, for RFM in step 3 ---
clean_cust = clean[clean['Customer ID'].notna()].copy()
print("\nRows with a Customer ID (for segmentation):", len(clean_cust),
      f"({len(clean_cust) / len(clean) * 100:.1f}% of cleaned rows)")

                                                rule  rows_removed  pct_of_original
0                                 Cancelled invoices         19494             1.83
1  Non-product codes (postage, fees, adjustments,...          4705             0.44
2                                      Quantity <= 0          3455             0.32
3                                         Price <= 0          2696             0.25

Starting rows: 1067371
Rows after cleaning: 1037021
Total removed: 30350 (2.8%)

Rows with a Customer ID (for segmentation): 802651 (77.4% of cleaned rows)


In [8]:
# ===== 1.5 Final checks and save the cleaned data =====

# --- Add a revenue column ---
# Every later step needs this, so compute it once here
clean['Revenue'] = clean['Quantity'] * clean['Price']
clean_cust['Revenue'] = clean_cust['Quantity'] * clean_cust['Price']

# --- Check for exact duplicate rows ---
# Same invoice, product, quantity, price and timestamp is likely a double entry
dupes = clean.duplicated().sum()
print("Exact duplicate rows:", dupes,
      f"({dupes / len(clean) * 100:.2f}%)")

# --- Check the date range ---
# Confirms the data covers what the documentation claims,
# and shows how incomplete December 2011 is
print("\nDate range:", clean['InvoiceDate'].min(), "to", clean['InvoiceDate'].max())

# Rows per month, to see the shape of the series before analysing it
# .dt.to_period('M') converts a timestamp to a year-month label
monthly_rows = clean['InvoiceDate'].dt.to_period('M').value_counts().sort_index()
print("\nRows in the first and last three months:")
print(monthly_rows.head(3))
print(monthly_rows.tail(3))

# --- Headline figures ---
print("\nTotal revenue: £{:,.0f}".format(clean['Revenue'].sum()))
print("Distinct products:", clean['StockCode'].nunique())
print("Distinct customers:", clean_cust['Customer ID'].nunique())
print("Distinct countries:", clean['Country'].nunique())

# --- Save both datasets ---
# CSV so later notebooks can load them without repeating the cleaning,
# and so Power BI can read them in step 7
clean.to_csv('cleaned_transactions.csv', index=False)
clean_cust.to_csv('cleaned_transactions_with_customer.csv', index=False)
print("\nSaved both files.")

Exact duplicate rows: 33664 (3.25%)

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00

Rows in the first and last three months:
InvoiceDate
2009-12    43824
2010-01    30499
2010-02    28146
Freq: M, Name: count, dtype: int64
InvoiceDate
2011-10    59068
2011-11    83092
2011-12    25031
Freq: M, Name: count, dtype: int64

Total revenue: £20,110,181
Distinct products: 4896
Distinct customers: 5852
Distinct countries: 43

Saved both files.


In [9]:
# ===== 1.6 Examine duplicates and measure their impact =====

# Look at a few examples, sorted so the pairs sit next to each other
dupe_rows = clean[clean.duplicated(keep=False)].sort_values(
    ['Invoice', 'StockCode', 'InvoiceDate'])
print("Sample of duplicated rows:")
print(dupe_rows.head(10)[['Invoice', 'StockCode', 'Description',
                          'Quantity', 'Price', 'InvoiceDate']])

# How many invoices are affected, and how much revenue sits in the duplicates
print("\nInvoices containing duplicates:", dupe_rows['Invoice'].nunique())

# keep='first' marks every copy after the first as a duplicate
extra = clean[clean.duplicated(keep='first')]
print("Revenue in the extra copies: £{:,.0f}".format(extra['Revenue'].sum()))
print("As a share of total revenue: {:.2f}%".format(
    extra['Revenue'].sum() / clean['Revenue'].sum() * 100))

Sample of duplicated rows:
    Invoice StockCode                       Description  Quantity  Price         InvoiceDate
379  489517     21491   SET OF THREE VINTAGE GIFT WRAPS         1   1.95 2009-12-01 11:34:00
391  489517     21491   SET OF THREE VINTAGE GIFT WRAPS         1   1.95 2009-12-01 11:34:00
365  489517     21821  GLITTER STAR GARLAND WITH BELLS          1   3.75 2009-12-01 11:34:00
386  489517     21821  GLITTER STAR GARLAND WITH BELLS          1   3.75 2009-12-01 11:34:00
363  489517     21912          VINTAGE SNAKES & LADDERS         1   3.75 2009-12-01 11:34:00
371  489517     21912          VINTAGE SNAKES & LADDERS         1   3.75 2009-12-01 11:34:00
394  489517     21912          VINTAGE SNAKES & LADDERS         1   3.75 2009-12-01 11:34:00
362  489517     21913    VINTAGE SEASIDE JIGSAW PUZZLES         1   3.75 2009-12-01 11:34:00
385  489517     21913    VINTAGE SEASIDE JIGSAW PUZZLES         1   3.75 2009-12-01 11:34:00
368  489517     22130  PARTY CONE CHRISTMAS